<a href="https://colab.research.google.com/github/nattaponm/Teach_Nowcast_Pysteps_TMD/blob/main/notebooks/01_hailstorm_radar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 1: พายุลูกเห็บเชียงของผ่านสายตาเรดาร์
### Reflectivity, Z–R relationship and hail contamination

⏱️ เวลาเรียนโดยประมาณ **60–75 นาที** | ต้องผ่าน: Notebook 0

---
## 🎯 วัตถุประสงค์การเรียนรู้
เมื่อจบ notebook นี้ นิสิตจะสามารถ
1. **อ่านและตีความ** แผนที่ reflectivity (dBZ) ของพายุฝนฟ้าคะนอง และบอกได้ว่าบริเวณใดอาจมีลูกเห็บ
2. **สร้างแผนที่ระดับตีพิมพ์** ทั้งแบบภาพรวมรัศมีเรดาร์และแบบซูมพื้นที่ศึกษา
3. **แปลง dBZ เป็นอัตราฝน** ด้วย Z–R relationship หลายชุด และอธิบายผลของ **hail cap**
4. **วิเคราะห์อนุกรมเวลา ณ จุด** (อ.เชียงของ) และฝนสะสมของเหตุการณ์

## 📖 1. แนวคิดและสมการ

### 1.1 การตีความค่า reflectivity
| dBZ | การตีความโดยประมาณ |
|---|---|
| < 20 | ฝนเบามาก / ละอองฝน |
| 20–35 | ฝนเบาถึงปานกลาง |
| 35–45 | ฝนหนัก (convective core) |
| 45–55 | ฝนหนักมาก **อาจมีลูกเห็บปน** |
| > 55 | **มีโอกาสสูงที่จะมีลูกเห็บ** (ลูกเห็บสะท้อนคลื่นได้แรงมาก) |

> ค่าเหล่านี้เป็นแนวทางเท่านั้น การยืนยันลูกเห็บต้องใช้ข้อมูลเพิ่ม เช่น echo top, VIL หรือ dual-polarization ($Z_{DR}$, $\rho_{HV}$)

### 1.2 Z–R relationship และ hail cap
$$
Z = a\,R^{\,b}
\quad\Longrightarrow\quad
R = \left(\frac{10^{\,\mathrm{dBZ}/10}}{a}\right)^{1/b}
$$

| ชุดสัมประสิทธิ์ | $a$ | $b$ | ลักษณะฝน |
|---|---|---|---|
| Marshall–Palmer | 200 | 1.6 | stratiform ทั่วไป (ค่า default ของวิชานี้) |
| WSR-88D convective | 300 | 1.4 | ฝนพาความร้อน (NEXRAD) |
| Rosenfeld tropical | 250 | 1.2 | ฝนเขตร้อน (warm rain) |

**Hail cap:** ลูกเห็บมีขนาดใหญ่และสะท้อนคลื่นแรงมาก ถ้าแปลง dBZ ที่มีลูกเห็บปนด้วยสมการฝน จะได้ $R$ สูงเกินจริงหลายเท่า จึงนิยม **จำกัดค่า dBZ สูงสุด** (เช่น 53–55 dBZ) ก่อนแปลง

$$
\mathrm{dBZ}_{\text{capped}} = \min(\mathrm{dBZ},\ \mathrm{dBZ}_{\text{cap}})
$$

### 1.3 ฝนสะสม
จากภาพทุก $\Delta t = 15$ นาที ฝนสะสมโดยประมาณคือ

$$
P\ (\mathrm{mm}) \approx \sum_{i} R_i \,\Delta t \quad (\Delta t \text{ หน่วยชั่วโมง} = 0.25\ \mathrm{h})
$$

ซึ่งยังหยาบอยู่ เพราะพายุเคลื่อนที่ระหว่างภาพ (จะปรับปรุงด้วย advection interpolation ใน NB2)

## ⚙️ 2. เตรียมสภาพแวดล้อมและข้อมูล

In [ ]:
#@title ⚙️ Setup: ติดตั้ง pysteps และดาวน์โหลดข้อมูล TMD (รันก่อนเสมอ ~2-3 นาทีใน Colab)
import os, sys, warnings
warnings.filterwarnings("ignore")

REPO = "Teach_Nowcast_Pysteps_TMD"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q pysteps==1.21.5 cartopy netCDF4 rasterio geopandas
    if not os.path.exists(REPO):
        !git clone -q --depth 1 https://github.com/nattaponm/{REPO}.git

# หาโฟลเดอร์หลักของ repo (ที่มี utils/ และ data/)
for ROOT in [REPO, ".", ".."]:
    if os.path.exists(os.path.join(ROOT, "utils", "tmd_tools.py")):
        break
ROOT = os.path.abspath(ROOT)
sys.path.insert(0, ROOT)
DATA = os.path.join(ROOT, "data")

import numpy as np
import matplotlib.pyplot as plt
import pysteps
from utils.data_io import load_radar_nc
from utils import mapplot as mp
from utils import tmd_tools as tt
mp.set_style()
mp.BASEMAP.update(provinces=True, rivers=True, lakes=False)   # จังหวัด + แม่น้ำโขง
mp.POINTS[:] = tt.POIS                                         # เรดาร์ เชียงของ เชียงแสน
print("✅ พร้อมใช้งาน | ข้อมูลอยู่ที่:", DATA)

In [ ]:
Z, meta = load_radar_nc(os.path.join(DATA, "tmd_cri_20200423_dbz_500m.nc"))
ts = meta["timestamps"]
print("ขนาด:", Z.shape, "| pixel:", round(meta["xpixelsize"]), "m | หน่วย:", meta["unit"])
print("ช่วงเวลา:", mp.tlabel(ts[0]), "→", mp.tlabel(ts[-1]))

BOX = tt.range_box(meta, 165)                                   # ภาพรวมรัศมีเรดาร์
CK_BOX = tt.zoom_box(meta, *tt.CHIANG_KHONG, half_width_km=45)  # ซูมรอบเชียงของ

## 🗺️ 3. ภาพรวมของเหตุการณ์ (ทุก 1 ชั่วโมง)

In [ ]:
idx = [0, 4, 8, 12]                          # 11:00, 12:00, 13:00, 14:00 UTC
fig, axs = mp.map_panels(1, 4, meta, panel_width=2.7, zoom=BOX)
for j, i in enumerate(idx):
    im = mp.radar_map(axs[j], Z[i], meta, units="dBZ", title=mp.tlabel(ts[i]), zoom=BOX,
                      gridline_labels=("left", "bottom") if j == 0 else ("bottom",),
                      north=(j == 0), scalebar=(j == 0), points=(j == 0))
    mp.add_range_rings(axs[j], meta, radii_km=(50, 100, 150), label=(j == 0))
mp.add_colorbar(fig, im, axs, label="Reflectivity at 2 km CAPPI (dBZ)")
mp.panel_labels(axs)
mp.save_figure(fig, "nb1_fig01_overview")
plt.show()

> 🔍 **สังเกต:** มีเซลล์พายุ (convective cells) หลายกลุ่มกระจัดกระจาย ไม่ใช่แถบฝนกว้างแบบในฟินแลนด์ (ชุดข้อมูลตัวอย่าง) กลุ่มพายุแรงอยู่ทางตะวันออกเฉียงเหนือของเรดาร์ ใกล้แม่น้ำโขงและอำเภอเชียงของ

## 🔎 4. ซูมพื้นที่ศึกษา: อ.เชียงของ (ความละเอียด 500 m)

In [ ]:
idx = [4, 5, 6]                               # 12:00, 12:15, 12:30 UTC
fig, axs = mp.map_panels(1, 3, meta, panel_width=3.2, zoom=CK_BOX)
for j, i in enumerate(idx):
    im = mp.radar_map(axs[j], Z[i], meta, units="dBZ", title=mp.tlabel(ts[i]), zoom=CK_BOX,
                      gridline_labels=("left", "bottom") if j == 0 else ("bottom",),
                      north=(j == 0))
    axs[j].contour(*np.meshgrid(meta["x1"] + meta["xpixelsize"] * (np.arange(Z.shape[2]) + 0.5),
                                meta["y2"] - meta["ypixelsize"] * (np.arange(Z.shape[1]) + 0.5)),
                   np.nan_to_num(Z[i]), levels=[50], colors="k", linewidths=0.8,
                   transform=mp.get_crs(meta), zorder=6)
mp.add_colorbar(fig, im, axs, label="Reflectivity (dBZ); black contour = 50 dBZ")
mp.panel_labels(axs)
mp.save_figure(fig, "nb1_fig02_chiang_khong_zoom")
plt.show()

## 📈 5. อนุกรมเวลา ณ อ.เชียงของ
ใช้ค่า **dBZ สูงสุดภายในรัศมี 5 km** รอบที่ว่าการอำเภอ (เพราะลูกเห็บตกเป็นพื้นที่ ไม่ใช่จุดเดียว และตำแหน่งเรดาร์อาจคลาดเคลื่อนเล็กน้อย)

In [ ]:
import matplotlib.dates as mdates
ck_max = tt.point_series(Z, meta, *tt.CHIANG_KHONG, radius_km=5, stat="max")

fig, ax = plt.subplots(figsize=(6, 3), layout="constrained")
ax.axhspan(45, 55, color="#fdd49e", alpha=0.6, label="45–55 dBZ: possible hail")
ax.axhspan(55, 65, color="#fc8d59", alpha=0.6, label="> 55 dBZ: hail likely")
ax.plot(ts, ck_max, "o-", color="k", ms=4)
ax.set_ylim(0, 65)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
mp.skill_axes(ax, xlabel="Time (UTC); local time = UTC + 7 h", ylabel="Max reflectivity (dBZ)",
              title="Chiang Khong (5-km radius)")
ax.legend(frameon=False, loc="lower right", fontsize=7)
mp.save_figure(fig, "nb1_fig03_chiang_khong_timeseries")
plt.show()
for t, v in zip(ts, ck_max):
    print(f"{mp.tlabel(t)}  {v:5.1f} dBZ")

## 🌧️ 6. จาก dBZ เป็นอัตราฝนและฝนสะสม
ฟังก์ชัน `tt.dbz_to_rainrate(dbz, a, b, hail_cap)` ใช้สมการในข้อ 1.2

In [ ]:
R = tt.dbz_to_rainrate(Z, a=200, b=1.6, hail_cap=55)          # Marshall–Palmer + hail cap
R_nocap = tt.dbz_to_rainrate(Z, a=200, b=1.6, hail_cap=None)
P = np.nansum(R, axis=0) * 0.25                                 # 13 ภาพ × 0.25 h
P[np.isnan(Z[0])] = np.nan
print(f"อัตราฝนสูงสุด: มี hail cap = {np.nanmax(R):.0f} mm/h | ไม่มี cap = {np.nanmax(R_nocap):.0f} mm/h")
print(f"ฝนสะสมสูงสุด 11:00–14:00 UTC: {np.nanmax(P):.1f} mm")

fig, axs = mp.map_panels(1, 2, meta, panel_width=3.8, zoom=BOX)
im1 = mp.radar_map(axs[0], R[5], meta, title=f"Rain rate, {mp.tlabel(ts[5])}", zoom=BOX)
mp.add_colorbar(fig, im1, axs[0], shrink=0.9)
im2 = mp.radar_map(axs[1], P, meta, ptype="depth", units="mm", title="Accumulation 11:00–14:00 UTC",
                   zoom=BOX, gridline_labels=("bottom",), north=False)
mp.add_colorbar(fig, im2, axs[1], shrink=0.9)
mp.panel_labels(axs)
mp.save_figure(fig, "nb1_fig04_rainrate_accumulation")
plt.show()

## 🔢 7. แปลงเป็น dBR (เตรียมสำหรับ nowcasting)
เหมือน Notebook 1 ของชุดข้อมูลตัวอย่าง: ค่าฝน (mm/h) มีการกระจายเบ้มาก จึงแปลงเป็น **dBR** ก่อนใช้ใน optical flow และ nowcasting

$$
\mathrm{dBR} = \begin{cases} 10\log_{10} R, & R \ge 0.1\ \mathrm{mm\,h^{-1}} \\[2pt] -15, & R < 0.1 \end{cases}
$$

In [ ]:
from pysteps.utils import transformation
R1, meta_r = load_radar_nc(os.path.join(DATA, "tmd_cri_20200423_rain_1km.nc"))   # ไฟล์ 1 km สำหรับ NB2–4
dbr, meta_dbr = transformation.dB_transform(R1, meta_r, threshold=0.1, zerovalue=-15.0)

r = R1[np.isfinite(R1) & (R1 >= 0.1)]
d = dbr[np.isfinite(dbr) & (dbr > -15)]
fig, axs = plt.subplots(1, 2, figsize=(7.5, 2.8), layout="constrained")
axs[0].hist(r, bins=np.linspace(0.1, 60, 40), color="#3a6ea5", edgecolor="white", lw=0.4)
axs[0].set_yscale("log")
mp.skill_axes(axs[0], xlabel="Rain rate (mm h$^{-1}$)", ylabel="Count (log)", title="(a) Rain rate")
axs[1].hist(d, bins=np.linspace(-10, 20, 40), color="#c0504d", edgecolor="white", lw=0.4)
mp.skill_axes(axs[1], xlabel="dBR", ylabel="Count", title="(b) dBR")
plt.show()
print(f"สัดส่วนพื้นที่มีฝน (≥ 0.1 mm/h) เฉลี่ย: {100*np.nanmean(R1 >= 0.1):.1f} % ของรัศมีเรดาร์")

> 🔍 พื้นที่ที่มีฝนมีเพียงไม่กี่ % ของรัศมีเรดาร์ (เซลล์พายุขนาดเล็ก) ซึ่งเป็นลักษณะของพายุฤดูร้อนในภาคเหนือ และจะทำให้การพยากรณ์ยากกว่ากรณีแถบฝนกว้าง

---
## 🧪 แบบฝึกหัด

### 🧪 แบบฝึกหัด 1.1: ติดตามพายุเข้าเชียงของ

สร้างรูป **1 แถว 4 panel** ซูมรอบเชียงของ (`zoom=CK_BOX`) ที่เวลา **11:45, 12:00, 12:15 และ 12:30 UTC** (index 3–6) พร้อม panel label และ colorbar ร่วม

**คำถาม:** พายุเคลื่อนเข้าเชียงของจากทิศใด? ความเร็วประมาณเท่าใด (km/h)? (ใช้ scale bar ช่วยวัด)

<details>
<summary><b>💡 คำใบ้ (คลิกเพื่อเปิด)</b></summary>

- `fig, axs = mp.map_panels(1, 4, meta, panel_width=2.6, zoom=CK_BOX)`
- `mp.radar_map(axs[j], Z[i], meta, units="dBZ", title=mp.tlabel(ts[i]), zoom=CK_BOX)`

</details>

In [ ]:
# ✍️ แบบฝึกหัด 1.1 — เขียนโค้ดของคุณที่นี่ (ลบ # หน้าบรรทัดแล้วเติมส่วนที่เป็น ...)
# fig, axs = mp.map_panels(1, 4, meta, panel_width=2.6, zoom=CK_BOX)
# for j, i in enumerate([3, 4, 5, 6]):
#     im = mp.radar_map(axs[j], Z[i], meta, units="dBZ", title=..., zoom=CK_BOX)
# mp.add_colorbar(fig, im, axs); mp.panel_labels(axs); plt.show()

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 1.1 (ลองทำเองก่อน! ดับเบิลคลิกเพื่อดูโค้ด แล้วกด ▶ เพื่อรัน)
fig, axs = mp.map_panels(1, 4, meta, panel_width=2.6, zoom=CK_BOX)
for j, i in enumerate([3, 4, 5, 6]):
    im = mp.radar_map(axs[j], Z[i], meta, units="dBZ", title=mp.tlabel(ts[i]), zoom=CK_BOX,
                      gridline_labels=("left", "bottom") if j == 0 else ("bottom",), north=(j == 0))
mp.add_colorbar(fig, im, axs, label="Reflectivity (dBZ)")
mp.panel_labels(axs)
mp.save_figure(fig, "nb1_ex11_storm_approach")
plt.show()
# พายุเคลื่อนจากทิศตะวันตก/ตะวันตกเฉียงใต้ไปทางตะวันออก ราว 30–45 km/h (ตรวจสอบเชิงปริมาณใน NB2 และ NB5)

### 🧪 แบบฝึกหัด 1.2: Z–R และ hail cap มีผลต่อฝนสะสมแค่ไหน?

คำนวณ **ฝนสะสม 11:00–14:00 UTC** (mm) ที่ **เชียงของ** (ค่าเฉลี่ยในรัศมี 5 km) สำหรับ Z–R ทั้ง 3 ชุดใน `tt.ZR_SETS` โดยทำทั้งกรณี **มี hail cap (55 dBZ)** และ **ไม่มี hail cap** แล้วสรุปเป็นตาราง (`pandas.DataFrame`)

**คำถาม:** การเลือก Z–R หรือ hail cap มีผลมากกว่ากัน? ถ้าจะใช้เรดาร์ประเมินฝนเพื่อเตือนน้ำท่วมฉับพลัน ควรตรวจสอบกับข้อมูลใด?

<details>
<summary><b>💡 คำใบ้ (คลิกเพื่อเปิด)</b></summary>

- วน loop `for name, (a, b) in tt.ZR_SETS.items():` และ `for cap in [55, None]:`
- `Rx = tt.dbz_to_rainrate(Z, a=a, b=b, hail_cap=cap)`
- ฝนสะสมที่จุด: `tt.point_series(Rx, meta, *tt.CHIANG_KHONG, radius_km=5, stat="mean").sum() * 0.25`

</details>

In [ ]:
# ✍️ แบบฝึกหัด 1.2 — เขียนโค้ดของคุณที่นี่ (ลบ # หน้าบรรทัดแล้วเติมส่วนที่เป็น ...)
# import pandas as pd
# rows = []
# for name, (a, b) in tt.ZR_SETS.items():
#     for cap in [55, None]:
#         Rx = tt.dbz_to_rainrate(Z, a=a, b=b, hail_cap=cap)
#         acc = ...
#         rows.append({"Z-R": name, "hail cap": cap, "accum_mm": acc})
# pd.DataFrame(rows)

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 1.2 (ลองทำเองก่อน! ดับเบิลคลิกเพื่อดูโค้ด แล้วกด ▶ เพื่อรัน)
import pandas as pd
rows = []
for name, (a, b) in tt.ZR_SETS.items():
    for cap in [55, None]:
        Rx = tt.dbz_to_rainrate(Z, a=a, b=b, hail_cap=cap)
        acc = tt.point_series(Rx, meta, *tt.CHIANG_KHONG, radius_km=5, stat="mean").sum() * 0.25
        peak = np.nanmax(tt.point_series(Rx, meta, *tt.CHIANG_KHONG, radius_km=5, stat="max"))
        rows.append({"Z-R": name, "a": a, "b": b, "hail cap (dBZ)": cap if cap else "none",
                     "CK accumulation (mm)": round(acc, 1), "CK peak rate (mm/h)": round(peak, 0)})
df_zr = pd.DataFrame(rows)
display(df_zr) if "display" in dir() else print(df_zr)

### 🧪 แบบฝึกหัด 1.3: ดัชนีความรุนแรงของพายุ: พื้นที่ ≥ 45 และ ≥ 55 dBZ

คำนวณ **พื้นที่ (km²)** ที่มี reflectivity ≥ 45 dBZ และ ≥ 55 dBZ ทั้งรัศมีเรดาร์ ทุกเวลา แล้วพล็อตกราฟเส้นเทียบกับเวลา (1 pixel = 0.25 km²)

**คำถาม:** พายุรุนแรงที่สุดช่วงเวลาใด? สอดคล้องกับช่วงที่ลูกเห็บตกที่เชียงของหรือไม่?

<details>
<summary><b>💡 คำใบ้ (คลิกเพื่อเปิด)</b></summary>

- `area45 = np.nansum(Z >= 45, axis=(1, 2)) * (meta["xpixelsize"] / 1000) ** 2`
- ใช้ `mdates.DateFormatter("%H:%M")` จัดรูปแบบแกนเวลา

</details>

In [ ]:
# ✍️ แบบฝึกหัด 1.3 — เขียนโค้ดของคุณที่นี่ (ลบ # หน้าบรรทัดแล้วเติมส่วนที่เป็น ...)
# px_km2 = (meta["xpixelsize"] / 1000) ** 2
# area45 = ...
# area55 = ...
# fig, ax = plt.subplots(figsize=(6, 3))
# ax.plot(ts, area45, label="≥ 45 dBZ"); ax.plot(ts, area55, label="≥ 55 dBZ")
# ax.legend(); plt.show()

In [ ]:
#@title ✅ เฉลยแบบฝึกหัด 1.3 (ลองทำเองก่อน! ดับเบิลคลิกเพื่อดูโค้ด แล้วกด ▶ เพื่อรัน)
px_km2 = (meta["xpixelsize"] / 1000) ** 2
with np.errstate(invalid="ignore"):
    area45 = np.sum(Z >= 45, axis=(1, 2)) * px_km2
    area55 = np.sum(Z >= 55, axis=(1, 2)) * px_km2
fig, ax = plt.subplots(figsize=(6, 3), layout="constrained")
ax.plot(ts, area45, "o-", ms=3, color="#fdae61", label="≥ 45 dBZ")
ax.plot(ts, area55, "s-", ms=3, color="#d7191c", label="≥ 55 dBZ")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
mp.skill_axes(ax, xlabel="Time (UTC)", ylabel="Area (km$^2$)", title="Area of intense echoes")
ax.legend(frameon=False)
mp.save_figure(fig, "nb1_ex13_intense_area")
plt.show()

---
## 📝 สรุป
- เคสนี้เป็น **พายุฝนฟ้าคะนองเป็นกลุ่มเซลล์** (multicell) ที่มี reflectivity > 55 dBZ ใกล้เชียงของ ซึ่งบ่งชี้ว่ามีลูกเห็บ
- การแปลง dBZ → ฝน ขึ้นกับ **Z–R** และ **hail cap** อย่างมาก ในงานวิจัยต้องระบุค่าที่ใช้และควรตรวจสอบกับเครื่องวัดน้ำฝน
- ภาพทุก 15 นาทีทำให้ฝนสะสมมีลักษณะเป็น "จุดกระโดด" ซึ่งจะแก้ใน NB2

### ❓ คำถามชวนคิด
1. ทำไมเรดาร์ที่ความสูง CAPPI 2 km อาจ "มองไม่เห็น" ฝนที่ตกจากเมฆต่ำบริเวณไกลเรดาร์?
2. ถ้าเรดาร์ใช้คลื่น C-band ความเข้มของพายุ "ด้านหลัง" เซลล์ที่แรงมากจะเป็นอย่างไร (attenuation)?

➡️ **บทถัดไป:** Notebook 2 — motion field ของพายุ, nowcast แรก และการเพิ่มความละเอียดเวลา 15 → 5 นาที

### 📚 เอกสารอ้างอิง
- Marshall, J. S., & Palmer, W. M. (1948). The distribution of raindrops with size. *J. Meteor.*, 5, 165–166.
- Fulton, R. A., et al. (1998). The WSR-88D rainfall algorithm. *Wea. Forecasting*, 13, 377–395.
- Rosenfeld, D., Wolff, D. B., & Atlas, D. (1993). General probability-matched relations between radar reflectivity and rain rate. *J. Appl. Meteor.*, 32, 50–72.
- Waldvogel, A., Federer, B., & Grimm, P. (1979). Criteria for the detection of hail cells. *J. Appl. Meteor.*, 18, 1521–1525.